In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

train = pd.read_csv('train.csv', low_memory=False)
store = pd.read_csv('store.csv')

df = pd.merge(train, store, on='Store', how='left')

print("Размер таблицы:", df.shape)
print("\nПропуски в данных:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nПервые 5 строк:\n", df.head())

df = df[(df['Open'] == 1) & (df['Sales'] > 0)].copy()

df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

cols_to_drop = ['Date', 'Customers', 'Open', 'PromoInterval']
df = df.drop(columns=cols_to_drop)

df['CompetitionDistance'] = df['CompetitionDistance'].fillna(df['CompetitionDistance'].median())
df = df.fillna(0)

df['StateHoliday'] = df['StateHoliday'].astype(str)
df = pd.get_dummies(df, drop_first=True)

X = df.drop('Sales', axis=1)
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

alphas = [0.1, 1.0, 10.0, 100.0]
best_alpha = None
best_rmse = float('inf')
best_model = None

print("\nРезультаты перебора параметров")
for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"Ridge (alpha={alpha}) -> RMSE: {rmse:.2f}")
    if rmse < best_rmse:
        best_rmse = rmse
        best_alpha = alpha
        best_model = model

print(f"\nЛучшая модель: Ridge с alpha = {best_alpha} (RMSE = {best_rmse:.2f})")

final_predictions = best_model.predict(X_test)

comparison = pd.DataFrame({
    'Реальные продажи': y_test.values[:10],
    'Прогноз модели': np.round(final_predictions[:10], 2)
})
print("\nПример прогноза")
print(comparison)


Размер таблицы: (1017209, 18)

Пропуски в данных:
 CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64

Первые 5 строк:
    Store  DayOfWeek        Date  Sales  Customers  Open  Promo StateHoliday  \
0      1          5  2015-07-31   5263        555     1      1            0   
1      2          5  2015-07-31   6064        625     1      1            0   
2      3          5  2015-07-31   8314        821     1      1            0   
3      4          5  2015-07-31  13995       1498     1      1            0   
4      5          5  2015-07-31   4822        559     1      1            0   

   SchoolHoliday StoreType Assortment  CompetitionDistance  \
0              1         c          a               1270.0   
1              1         a          a                570.0   
2              1         a          a    